# Módulo 02 · Aula 4 — Juntando e limpando dados

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Existe uma estatística repetida à exaustão na área: **80% do tempo de um projeto de
dados é gasto preparando os dados**. O número é impreciso, mas a mensagem é verdadeira —
e é justamente essa parte que os cursos costumam pular.

Esta aula é sobre ela. Duas habilidades:

- **juntar** tabelas que vieram de fontes diferentes (`concat`, `merge`);
- **limpar** uma base real: valores faltantes, duplicatas, tipos errados, categorias
  inconsistentes e valores impossíveis.

**Tempo estimado:** 80 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "02_Manipulacao_Dados"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")
indicadores = pd.read_csv("../data/indicadores_macro.csv", parse_dates=["data"])

print("acoes      :", acoes.shape)
print("empresas   :", empresas.shape)
print("indicadores:", indicadores.shape)

## 1. `concat`: empilhar

Quando as tabelas têm **as mesmas colunas** e você quer juntar as linhas, o comando é
`concat`. O caso típico: um arquivo por mês, um por ano, um por filial.

In [ ]:
petr = acoes[acoes["ticker"] == "PETR4"].head(3)
vale = acoes[acoes["ticker"] == "VALE3"].head(3)

empilhado = pd.concat([petr, vale], ignore_index=True)
empilhado[["data", "ticker", "fechamento"]]

> `ignore_index=True` recria o índice como 0, 1, 2... Sem ele, os índices originais são
> preservados e você acaba com rótulos repetidos — que depois causam confusão em `.loc`.

## 2. `merge`: cruzar por uma chave

`concat` empilha; `merge` **cruza**. Ele combina duas tabelas que compartilham uma
coluna em comum — a **chave** — trazendo as colunas de uma para a outra. É o `PROCV` do
Excel, ou o `JOIN` do SQL.

Nossa tabela `empresas` tem informações cadastrais que não estão na tabela de preços:

In [ ]:
empresas

In [ ]:
acoes_completo = acoes.merge(empresas, on="ticker", how="left")

print("Antes :", acoes.shape)
print("Depois:", acoes_completo.shape, "  <- mesmas linhas, 4 colunas a mais")
acoes_completo.head(3)

### Os quatro tipos de junção

O parâmetro `how` decide **quais linhas sobrevivem** quando a chave não existe nos dois
lados:

| `how` | Mantém |
|---|---|
| `"inner"` | só as chaves presentes **nas duas** tabelas (padrão) |
| `"left"` | **todas** as da esquerda; o que não casar vira `NaN` |
| `"right"` | todas as da direita |
| `"outer"` | todas as duas, casando o que der |

Na prática, `left` e `inner` respondem por quase tudo. `left` quando a tabela da
esquerda é a base e você está apenas enriquecendo; `inner` quando só interessam os
registros completos.

In [ ]:
# Uma tabela de exemplo com um ticker que NÃO existe no cadastro
alvos = pd.DataFrame({
    "ticker": ["PETR4", "VALE3", "XPTO3"],
    "preco_alvo": [40.00, 75.00, 12.00],
})

print("--- inner: XPTO3 desaparece ---")
print(alvos.merge(empresas[["ticker", "empresa"]], on="ticker", how="inner"))

print("\n--- left: XPTO3 fica, com empresa NaN ---")
print(alvos.merge(empresas[["ticker", "empresa"]], on="ticker", how="left"))

### Conferindo o resultado do merge

**Todo merge deve ser conferido.** Dois erros são silenciosos e caros:

1. **perder linhas** sem perceber (chave escrita de forma diferente nas duas tabelas);
2. **multiplicar linhas** sem perceber (a chave se repete no lado direito, e cada linha
   da esquerda casa com várias).

Duas ferramentas evitam os dois:

In [ ]:
# indicator=True cria uma coluna dizendo de onde veio cada linha
conferencia = alvos.merge(
    empresas[["ticker", "empresa"]], on="ticker", how="outer", indicator=True
)
print(conferencia[["ticker", "empresa", "_merge"]])
print()
print(conferencia["_merge"].value_counts())

In [ ]:
# validate declara qual é a relação esperada; o pandas ERRA se ela não se confirmar.
# "m:1" = muitas linhas de preços para um registro de cadastro.
resultado = acoes.merge(empresas, on="ticker", how="left", validate="m:1")
print("Junção validada:", resultado.shape)

# Se o cadastro tivesse tickers duplicados, a linha abaixo levantaria um erro
cadastro_duplicado = pd.concat([empresas, empresas.head(1)], ignore_index=True)
try:
    acoes.merge(cadastro_duplicado, on="ticker", how="left", validate="m:1")
except Exception as erro:
    print("MergeError:", erro)

> **Ganhe o hábito de conferir o `shape` antes e depois de todo merge.** Se o número
> de linhas mudou e você não esperava por isso, pare e investigue antes de seguir. Um
> merge que duplicou linhas contamina todas as somas e médias que vierem depois.

### Chaves com nomes diferentes

Nem sempre a coluna se chama igual nas duas tabelas. Use `left_on` e `right_on`:

In [ ]:
outro_cadastro = empresas.rename(columns={"ticker": "codigo_negociacao"})

acoes.head(3).merge(
    outro_cadastro[["codigo_negociacao", "setor"]],
    left_on="ticker",
    right_on="codigo_negociacao",
    how="left",
)[["data", "ticker", "codigo_negociacao", "setor"]]

### Cruzando frequências diferentes

Um caso muito comum em finanças: preços são **diários** e indicadores macro são
**mensais**. A chave precisa ser construída antes do merge.

In [ ]:
indicadores.head(3)

In [ ]:
# Criamos, nas duas tabelas, uma coluna de período mensal para servir de chave
acoes_mensal = acoes.copy()
acoes_mensal["ano_mes"] = acoes_mensal["data"].dt.to_period("M")

indicadores_mensal = indicadores.copy()
indicadores_mensal["ano_mes"] = indicadores_mensal["data"].dt.to_period("M")

junto = acoes_mensal.merge(
    indicadores_mensal.drop(columns="data"),
    on="ano_mes",
    how="left",
    validate="m:1",
)

junto[["data", "ticker", "fechamento", "ipca_mes_pct", "selic_mes_pct", "dolar_medio"]].head()

## 3. Limpeza de dados

Agora a base suja. `clientes_corretora.csv` é um cadastro **fictício** de uma corretora,
construído com os defeitos que aparecem em bases reais. Vamos limpá-la do começo ao fim.

In [ ]:
clientes = pd.read_csv("../data/clientes_corretora.csv")

print(clientes.shape)
clientes.head()

In [ ]:
clientes.info()

Leia o `info()` com desconfiança e anote os problemas:

- `patrimonio_investido` está como **`object`** — deveria ser número;
- `data_cadastro` está como **`object`** — deveria ser data;
- `idade`, `cidade` e `perfil_investidor` têm **menos valores** que as 400 linhas;
- `ativo` também é texto, e vimos que tem quatro valores diferentes para duas situações.

E o `describe()` mostra mais um:

In [ ]:
clientes.describe()

Idade mínima **negativa** e máxima de **250 anos**. São valores impossíveis, não
extremos — a diferença importa e voltaremos a ela.

### O plano

1. valores faltantes
2. duplicatas
3. tipos incorretos
4. categorias inconsistentes
5. valores impossíveis

Trabalharemos sempre sobre uma **cópia**, preservando o original para poder comparar.

In [ ]:
limpo = clientes.copy()

### 3.1 Valores faltantes

Primeiro, medir. Nunca trate o que você não mediu.

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": limpo.isna().sum(),
    "percentual": (limpo.isna().mean() * 100).round(1),
})
faltantes[faltantes["faltantes"] > 0]

Agora a decisão. Existem três caminhos, e escolher é uma decisão **analítica**, não
técnica:

| Estratégia | Quando faz sentido | Risco |
|---|---|---|
| **Remover a linha** (`dropna`) | poucos casos, e a ausência é aleatória | perder informação; enviesar se a ausência não for aleatória |
| **Preencher** (`fillna`) | você tem uma estimativa defensável | inventar dado; reduzir artificialmente a variabilidade |
| **Manter como está** | a ausência é informativa por si só | precisa lembrar disso em toda conta seguinte |

> **Atenção:** A pergunta que decide é: **por que esse dado está faltando?** Se os
> clientes de maior patrimônio são justamente os que não informam, preencher com a média
> puxa a distribuição para baixo e distorce toda a análise. "Faltante" quase nunca é
> aleatório.

Vamos aplicar uma escolha diferente para cada coluna, e deixar o motivo escrito.

In [ ]:
# idade: preenchemos com a MEDIANA. Ela é menos sensível a valores extremos que a média,
# e a proporção de faltantes é pequena.
mediana_idade = limpo["idade"].median()
limpo["idade"] = limpo["idade"].fillna(mediana_idade)
print(f"Idade preenchida com a mediana: {mediana_idade:.0f} anos")

# cidade: preenchemos com um rótulo explícito. Inventar uma cidade seria pior;
# marcar como desconhecida preserva a informação de que não sabemos.
limpo["cidade"] = limpo["cidade"].fillna("Não informado")

# perfil_investidor: MANTEMOS o faltante. O perfil vem de um questionário
# regulatório; não respondê-lo é um fato sobre o cliente, não um erro de digitação.
print("\nFaltantes restantes:")
print(limpo.isna().sum()[limpo.isna().sum() > 0])

Sobrou `patrimonio_investido`. Vamos tratá-lo depois de corrigir o tipo — não dá para
calcular a mediana de uma coluna de texto.

### 3.2 Duplicatas

In [ ]:
print("Linhas totalmente duplicadas:", limpo.duplicated().sum())

# Mais importante: duplicatas na CHAVE. Cada cliente deveria aparecer uma vez.
print("id_cliente repetidos:", limpo["id_cliente"].duplicated().sum())

limpo[limpo.duplicated(keep=False)].sort_values("id_cliente").head(4)

In [ ]:
antes = len(limpo)
limpo = limpo.drop_duplicates()
print(f"{antes} -> {len(limpo)} linhas ({antes - len(limpo)} duplicatas removidas)")

# Conferindo que a chave ficou única
print("id_cliente únicos?", limpo["id_cliente"].is_unique)

> `drop_duplicates()` sem argumentos remove linhas **idênticas em todas as colunas**.
> Para deduplicar por chave — mantendo, por exemplo, o registro mais recente — use
> `drop_duplicates(subset="id_cliente", keep="last")`. Ordene antes, para saber o que
> "last" significa.

### 3.3 Tipos incorretos

**Dinheiro salvo como texto.** Olhe os valores da coluna:

In [ ]:
print(limpo["patrimonio_investido"].head(8).tolist())

Há três formatos misturados: número puro, texto no formato brasileiro
(`"R$ 412.666,21"`) e ausentes. Para virar número, é preciso remover `R$`, remover o
ponto de milhar e trocar a vírgula decimal por ponto.

In [ ]:
def texto_para_numero(coluna):
    """
    Converte uma coluna com valores monetários em formato brasileiro para float.

    Trata "R$ 1.234,56", "1234.56" e valores ausentes. O que não for
    conversível vira NaN, em vez de derrubar a execução.
    """
    texto = coluna.astype(str).str.strip()
    texto = texto.str.replace("R$", "", regex=False)
    texto = texto.str.replace(".", "", regex=False)    # ponto de milhar
    texto = texto.str.replace(",", ".", regex=False)   # vírgula decimal
    return pd.to_numeric(texto, errors="coerce")


limpo["patrimonio_investido"] = texto_para_numero(limpo["patrimonio_investido"])

print(limpo["patrimonio_investido"].dtype)
print(limpo["patrimonio_investido"].describe().round(2))

> **Cuidado com a ordem das substituições.** Se trocássemos a vírgula por ponto antes
> de remover o ponto de milhar, `"1.234,56"` viraria `"1.234.56"` — que não é número
> nenhum. Faça o teste depois, com um valor conhecido.
>
> E note o `errors="coerce"`: o que não converter vira `NaN` em vez de interromper o
> programa. Depois é obrigatório **conferir quantos viraram NaN** — se forem muitos, a
> regra de conversão está errada.

In [ ]:
# Agora sim dá para tratar os faltantes de patrimônio
print("Faltantes em patrimonio_investido:", limpo["patrimonio_investido"].isna().sum())

# Decisão: manter como faltante. Patrimônio é a variável central da análise;
# preenchê-la com a média criaria clientes fictícios no meio da distribuição.

In [ ]:
# Datas em dois formatos diferentes
print(limpo["data_cadastro"].head(6).tolist())

In [ ]:
# Estratégia: tentar cada formato explicitamente e combinar os resultados.
# É mais previsível do que deixar o pandas adivinhar linha a linha.
formato_iso = pd.to_datetime(limpo["data_cadastro"], format="%Y-%m-%d", errors="coerce")
formato_br = pd.to_datetime(limpo["data_cadastro"], format="%d/%m/%Y", errors="coerce")

limpo["data_cadastro"] = formato_iso.fillna(formato_br)

print("Tipo:", limpo["data_cadastro"].dtype)
print("Não convertidas:", limpo["data_cadastro"].isna().sum())
print(limpo["data_cadastro"].min(), "->", limpo["data_cadastro"].max())

> **Atenção — Nunca deixe o pandas adivinhar formatos de data sem conferir.** `01/02/2024`
> é 1º de fevereiro para um brasileiro e 2 de janeiro para um americano. O pandas escolhe
> um, e a escolha errada não gera erro nenhum — só resultados errados. Sempre passe
> `format=` explicitamente, ou `dayfirst=True`, e confira o intervalo resultante.

In [ ]:
# A coluna "ativo": quatro valores para duas situações
print(limpo["ativo"].value_counts())

mapa_ativo = {"sim": True, "nao": False, "1": True, "0": False}
limpo["ativo"] = limpo["ativo"].astype(str).str.strip().str.lower().map(mapa_ativo)

print()
print(limpo["ativo"].value_counts(dropna=False))
print("Tipo:", limpo["ativo"].dtype)

> `.map()` com dicionário é a ferramenta certa para recodificar categorias. O que não
> estiver no dicionário vira `NaN` — o que é bom: revela imediatamente valores que você
> não previu. Confira sempre se apareceu algum.

### 3.4 Categorias inconsistentes

In [ ]:
print(limpo["perfil_investidor"].value_counts(dropna=False))

In [ ]:
limpo["perfil_investidor"] = limpo["perfil_investidor"].str.strip().str.title()
print(limpo["perfil_investidor"].value_counts(dropna=False))

In [ ]:
# Acentuação inconsistente nas cidades
print(sorted(limpo["cidade"].unique()))

In [ ]:
def padronizar_texto(coluna):
    """
    Padroniza uma coluna de texto para servir de categoria.

    Remove espaços nas pontas, remove acentos e coloca tudo em maiúsculas,
    de modo que "São Paulo", "sao paulo" e " SAO PAULO " virem um valor só.
    """
    return (
        coluna.astype(str)
        .str.strip()
        .str.normalize("NFKD")                          # separa a letra do acento
        .str.encode("ascii", errors="ignore")           # descarta os acentos
        .str.decode("utf-8")
        .str.upper()
    )


antes_cidades = clientes["cidade"].nunique()
limpo["cidade"] = padronizar_texto(limpo["cidade"])

print(sorted(limpo["cidade"].unique()))
print(f"\nDe {antes_cidades} valores distintos para {limpo['cidade'].nunique()}.")

Todas aquelas variações — `"São Paulo"`, `"SÃO PAULO"`, `"sao paulo"` — eram a mesma
cidade contada várias vezes. Em um `groupby("cidade")` elas apareceriam como linhas
distintas, e ninguém notaria o erro só olhando o resultado.

> Padronizar em maiúsculas sem acento tem um custo: o relatório final mostrará
> `"SAO PAULO"`. Em um projeto real, o padrão é manter **duas colunas** — uma
> padronizada, para agrupar e cruzar, e outra com o nome bonito, para exibir. Aqui
> simplificamos.

### 3.5 Valores impossíveis

Voltemos à idade.

In [ ]:
print(limpo["idade"].describe().round(1))
print()
print("Idades fora de [18, 110]:")
print(limpo.loc[~limpo["idade"].between(18, 110), ["id_cliente", "nome", "idade"]])

**Valor impossível não é a mesma coisa que valor extremo.** Um cliente de 95 anos é
plausível e deve ficar. Um de −3 ou de 250 não existe: é erro de digitação ou código de
sistema (às vezes `999` significa "não informado"). O primeiro é dado; o segundo é lixo.

O tratamento correto é transformar o impossível em **faltante** — porque é isso que ele
é — e então decidir o que fazer com o faltante.

In [ ]:
limpo.loc[~limpo["idade"].between(18, 110), "idade"] = np.nan

print("Idades inválidas restantes:", limpo["idade"].isna().sum())

limpo["idade"] = limpo["idade"].fillna(limpo["idade"].median()).astype(int)
print(limpo["idade"].describe().round(1))

> Repare na sintaxe `limpo.loc[mascara, "coluna"] = valor`. **Sempre use `.loc` para
> atribuir**. A forma `limpo[mascara]["idade"] = np.nan` parece equivalente, mas
> frequentemente altera uma cópia temporária e não a tabela — é a origem do aviso
> `SettingWithCopyWarning`, e do bug silencioso de "a alteração não pegou".

### 3.6 Conferindo o resultado

In [ ]:
limpo.info()

In [ ]:
limpo.describe().round(2)

In [ ]:
# Comparando antes e depois
comparacao = pd.DataFrame({
    "antes": [
        len(clientes),
        clientes["cidade"].nunique(),
        clientes["perfil_investidor"].nunique(),
        clientes["ativo"].nunique(),
        clientes.isna().sum().sum(),
    ],
    "depois": [
        len(limpo),
        limpo["cidade"].nunique(),
        limpo["perfil_investidor"].nunique(),
        limpo["ativo"].nunique(),
        limpo.isna().sum().sum(),
    ],
}, index=["linhas", "cidades distintas", "perfis distintos",
          "valores de 'ativo'", "células vazias"])

comparacao

As células vazias **aumentaram**, e isso está correto: convertemos idades impossíveis e
patrimônios ilegíveis em `NaN` explícitos. Antes, o problema estava escondido dentro de
valores que pareciam válidos. Um `NaN` visível é muito melhor que um −3 disfarçado de
idade.

### 3.7 Agora as análises fazem sentido

In [ ]:
resumo = limpo.groupby("perfil_investidor").agg(
    clientes=("id_cliente", "count"),
    idade_media=("idade", "mean"),
    patrimonio_mediano=("patrimonio_investido", "median"),
    aporte_medio=("aporte_mensal", "mean"),
    taxa_ativos=("ativo", "mean"),
).round(2)

resumo

Compare com o que teríamos obtido antes da limpeza: seis linhas de perfil em vez de
três, idades puxadas por um cliente de 250 anos, e `patrimonio_investido` sequer
agregável por ser texto. **A limpeza não é preparação para a análise — ela é parte da
análise.**

## 4. Um roteiro para levar com você

Cole isto na parede. Ao receber qualquer base nova:

```
1. OLHAR       shape · head · tail · info · describe
2. TIPOS       cada coluna está no tipo certo?
               (números como object = alerta vermelho)
3. FALTANTES   quanto falta, em quais colunas, e POR QUE falta?
4. DUPLICATAS  linhas idênticas? chave repetida?
5. CATEGORIAS  value_counts() em cada coluna de texto —
               procure o mesmo valor escrito de formas diferentes
6. IMPOSSÍVEIS min e max fazem sentido no mundo real?
7. CONFERIR    refazer info/describe e comparar com o esperado
8. REGISTRAR   deixar escrito, no notebook, cada decisão e o motivo
```

O passo 8 é o mais negligenciado e o mais importante. Toda escolha de limpeza é uma
premissa da sua análise. Se não estiver escrita, ninguém — inclusive você, daqui a um
mês — vai conseguir avaliar se o resultado é confiável.

## 5. Recapitulando

- `concat` empilha tabelas de mesma estrutura; `merge` cruza tabelas por uma **chave**.
- `how` define quem sobrevive: `inner` (só o comum), `left` (tudo da esquerda),
  `right`, `outer`. Confira `shape` antes e depois, e use `validate=` e
  `indicator=True`.
- Faltantes: meça com `isna().sum()`, e escolha entre remover, preencher ou manter — com
  base no **motivo** da ausência.
- Duplicatas: `duplicated()` e `drop_duplicates()`; verifique também a unicidade da
  chave.
- Tipos: `pd.to_numeric(..., errors="coerce")` e `pd.to_datetime(..., format=...)`.
  Sempre confira quantos viraram `NaN`.
- Categorias: `.str.strip().str.title()`, remoção de acentos, `.map()` com dicionário.
- Valores impossíveis viram `NaN` — e aí você decide. Impossível ≠ extremo.
- Atribua sempre com `.loc[mascara, "coluna"] = valor`.

**Fim do módulo 02.** Faça a `lista_02_manipulacao.ipynb` antes de seguir para
visualização.